In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**SINGLE FILE VALIDATION**

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("TeamDataInvestigation").getOrCreate()

# Set correct Kaggle dataset input path
data_path = "/kaggle/input/datasets/praneedataanalytics/esport-analysis"

**teams.csv**

In [11]:
# Load teams.csv safely
df_teams = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_path}/teams.csv")

# Verify it works
df_teams.show(5)

+-------+-------------+------+---------+
|team_id|    team_name|region|    coach|
+-------+-------------+------+---------+
|   TM01|Pixel Raiders|  APAC|   N. Rao|
|   TM02|Critical Nova|    EU| L. Meyer|
|   TM03| Quantum Five|    NA|J. Brooks|
|   TM04|  Night Shift|  APAC|  S. Iyer|
|   TM05|Echo Protocol|    EU| M. Klein|
+-------+-------------+------+---------+
only showing top 5 rows


In [12]:
print("=== 1. Column Names and Data Types ===")
df_teams.printSchema()

print(f"\n=== 2. Total Number of Rows ===")
print(f"Total rows: {df_teams.count()}")

print(f"\n=== 3. Sample of Actual Records ===")
df_teams.show(5, truncate=False)

=== 1. Column Names and Data Types ===
root
 |-- team_id: string (nullable = true)
 |-- team_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- coach: string (nullable = true)


=== 2. Total Number of Rows ===
Total rows: 8

=== 3. Sample of Actual Records ===
+-------+-------------+------+---------+
|team_id|team_name    |region|coach    |
+-------+-------------+------+---------+
|TM01   |Pixel Raiders|APAC  |N. Rao   |
|TM02   |Critical Nova|EU    |L. Meyer |
|TM03   |Quantum Five |NA    |J. Brooks|
|TM04   |Night Shift  |APAC  |S. Iyer  |
|TM05   |Echo Protocol|EU    |M. Klein |
+-------+-------------+------+---------+
only showing top 5 rows


In [13]:
print(f"\n=== 4. Number of Distinct team_id Values ===")
distinct_teams = df_teams.select("team_id").distinct().count()
print(f"Distinct team_id count: {distinct_teams}")


=== 4. Number of Distinct team_id Values ===
Distinct team_id count: 8


**#Null checking**

In [14]:
# Build expressions to count anomalies for each column
missing_checks = []
columns_to_check = ["team_id", "team_name", "region", "coach"]

for c in columns_to_check:
  missing_checks.append(
      F.sum(
          F.when(
              F.col(c).isNull()
              | (F.col(c) == "")
              | (F.length(F.trim(F.col(c))) == 0),
              1,
          ).otherwise(0)
      ).alias(f"{c}_missing_count")
  )

In [15]:
# Execute the check on the DataFrame
df_teams.select(missing_checks).show(truncate=False)

+---------------------+-----------------------+--------------------+-------------------+
|team_id_missing_count|team_name_missing_count|region_missing_count|coach_missing_count|
+---------------------+-----------------------+--------------------+-------------------+
|0                    |0                      |0                   |1                  |
+---------------------+-----------------------+--------------------+-------------------+



In [16]:
# Filter and display the record where the coach column is null, empty, or whitespace-only
df_teams.filter(
    F.col("coach").isNull()
    | (F.col("coach") == "")
    | (F.length(F.trim(F.col("coach"))) == 0)
).select("team_id", "team_name", "region", "coach").show(truncate=False)

+-------+------------+------+-----+
|team_id|team_name   |region|coach|
+-------+------------+------+-----+
|TM07   |Blue Phoenix|APAC  |NULL |
+-------+------------+------+-----+



In [17]:
# Here I could see that only the coach is NULL, remaining col values are perfectly fine
# So, I am not dropping this row/ filling the coach col with some value
# Instead I am leaving this as NULL, and will use this record for analysis that does not
# require any coach column

**Duplicate Records Handling**

In [18]:
# Stage A: Check for duplicate team_id values
df_teams.groupBy("team_id").count().filter(F.col("count") > 1).show(
    truncate=False
)

+-------+-----+
|team_id|count|
+-------+-----+
+-------+-----+



In [19]:
# Stage B: Check for exact duplicate rows across all columns
# Compare total row count with the row count after dropping exact duplicates
total_rows = df_teams.count()
unique_rows = df_teams.dropDuplicates().count()

print(f"Total rows: {total_rows}")
print(f"Unique rows (after dropping duplicates): {unique_rows}")
print(f"Exact duplicate rows found: {total_rows - unique_rows}")

Total rows: 8
Unique rows (after dropping duplicates): 8
Exact duplicate rows found: 0


**Standardization**

In [20]:
df_teams.select("region").distinct().show(truncate=False)

+------+
|region|
+------+
|NA    |
|SA    |
|APAC  |
|EU    |
+------+



In [21]:
df_teams.groupBy("region").count().show(truncate=False)

+------+-----+
|region|count|
+------+-----+
|NA    |2    |
|SA    |1    |
|APAC  |3    |
|EU    |2    |
+------+-----+



In [22]:
# In the teams.csv, I could not find any issues with the data as such

**players.csv**

In [23]:
# 1. Load players.csv into a Spark DataFrame
df_players = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_path}/players.csv")

In [24]:
print("=== 1. Column Names and Data Types ===")
df_players.printSchema()

=== 1. Column Names and Data Types ===
root
 |-- player_id: string (nullable = true)
 |-- player_name: string (nullable = true)
 |-- team_id: string (nullable = true)
 |-- role: string (nullable = true)
 |-- join_date: date (nullable = true)
 |-- status: string (nullable = true)



In [25]:
print(f"\n=== 2. Total Number of Rows ===")
print(f"Total rows: {df_players.count()}")

print(f"\n=== 3. Sample of Actual Records ===")
df_players.show(5, truncate=False)


=== 2. Total Number of Rows ===
Total rows: 80

=== 3. Sample of Actual Records ===
+---------+-----------+-------+----------+----------+----------+
|player_id|player_name|team_id|role      |join_date |status    |
+---------+-----------+-------+----------+----------+----------+
|PL001    |Zero_1     |TM01   |flex      |2023-12-31|inactive  |
|PL002    |Zero_2     |TM04   |sentinel  |2023-02-05|substitute|
|PL003    |Mako_3     |TM08   |support   |2025-08-08|active    |
|PL004    |Rift_4     |TM07   |support   |2025-01-04|Active    |
|PL005    |Zero_5     |TM07   |controller|2023-09-20|Active    |
+---------+-----------+-------+----------+----------+----------+
only showing top 5 rows


In [26]:
print(f"\n=== 4. Number of Distinct player_id Values ===")
distinct_players = df_players.select("player_id").distinct().count()
print(f"Distinct player_id count: {distinct_players}")


=== 4. Number of Distinct player_id Values ===
Distinct player_id count: 80


**Null handling**

In [27]:
# Build expressions to count anomalies for each column in df_players
missing_checks_players = []
columns_to_check_players = [
    "player_id",
    "player_name",
    "team_id",
    "role",
    "join_date",
    "status",
]

In [28]:
for c in columns_to_check_players:
  # For the date column, checking isNull() is the primary requirement
  if c == "join_date":
    condition = F.col(c).isNull()
  else:
    condition = (
        F.col(c).isNull()
        | (F.col(c) == "")
        | (F.length(F.trim(F.col(c))) == 0)
    )

  missing_checks_players.append(
      F.sum(F.when(condition, 1).otherwise(0)).alias(f"{c}_missing_count")
  )

# Execute the check on df_players
df_players.select(missing_checks_players).show(truncate=False)

+-----------------------+-------------------------+---------------------+------------------+-----------------------+--------------------+
|player_id_missing_count|player_name_missing_count|team_id_missing_count|role_missing_count|join_date_missing_count|status_missing_count|
+-----------------------+-------------------------+---------------------+------------------+-----------------------+--------------------+
|0                      |0                        |0                    |1                 |0                      |0                   |
+-----------------------+-------------------------+---------------------+------------------+-----------------------+--------------------+



In [29]:
# Filter and display the record where the role column is null, empty, or whitespace-only
df_players.filter(
    F.col("role").isNull()
    | (F.col("role") == "")
    | (F.length(F.trim(F.col("role"))) == 0)
).select(
    "player_id", "player_name", "team_id", "role", "join_date", "status"
).show(
    truncate=False
)

+---------+-----------+-------+----+----------+--------+
|player_id|player_name|team_id|role|join_date |status  |
+---------+-----------+-------+----+----------+--------+
|PL021    |Nova_21    |TM01   |NULL|2026-01-25|inactive|
+---------+-----------+-------+----+----------+--------+



In [30]:
# AS only the role is NULL, I want to leave it as null becuase remaining cols are good
# We can use this row, when we have any analytics to be performed which does not
# need any role

**de duplication**

In [31]:
# Check for duplicate player_id values
df_players.groupBy("player_id").count().filter(F.col("count") > 1).show(
    truncate=False
)

+---------+-----+
|player_id|count|
+---------+-----+
+---------+-----+



In [32]:
# Check for exact duplicate rows across all columns in df_players
total_player_rows = df_players.count()
unique_player_rows = df_players.dropDuplicates().count()

print(f"Total rows: {total_player_rows}")
print(f"Unique rows (after dropping duplicates): {unique_player_rows}")
print(f"Exact duplicate rows found: {total_player_rows - unique_player_rows}")

Total rows: 80
Unique rows (after dropping duplicates): 80
Exact duplicate rows found: 0


**standardization**

In [33]:
print("=== Role Value Distribution ===")
df_players.groupBy("role").count().orderBy(F.col("count").desc()).show(
    truncate=False
)

print("=== Status Value Distribution ===")
df_players.groupBy("status").count().orderBy(F.col("count").desc()).show(
    truncate=False
)

=== Role Value Distribution ===
+----------+-----+
|role      |count|
+----------+-----+
|duelist   |17   |
|controller|14   |
|support   |14   |
|sentinel  |13   |
|flex      |12   |
|Flex      |9    |
|NULL      |1    |
+----------+-----+

=== Status Value Distribution ===
+----------+-----+
|status    |count|
+----------+-----+
|Active    |22   |
|inactive  |21   |
|active    |20   |
|substitute|17   |
+----------+-----+



In [34]:
# 1. Apply standardization (converting role and status to lowercase)
df_players_standardized = df_players.withColumn(
    "role", F.lower(F.col("role"))
).withColumn("status", F.lower(F.col("status")))

In [35]:
# 2. Validation Check 1: Invariance Test (Row Count should remain 80)
initial_count = df_players.count()
cleaned_count = df_players_standardized.count()

print(f"=== Validation 1: Row Count Invariance ===")
print(f"Before cleaning: {initial_count}")
print(f"After cleaning:  {cleaned_count}")
assert (
    initial_count == cleaned_count
), "Warning: Row count changed during standardization!"
print("Status: PASSED (Row counts match)\n")

=== Validation 1: Row Count Invariance ===
Before cleaning: 80
After cleaning:  80
Status: PASSED (Row counts match)



In [36]:
# 3. Validation Check 2: Consolidation Test (Check new distributions)
print("=== Validation 2: Standardized Role Distribution ===")
df_players_standardized.groupBy("role").count().orderBy(
    F.col("count").desc()
).show(truncate=False)

print("=== Validation 2: Standardized Status Distribution ===")
df_players_standardized.groupBy("status").count().orderBy(
    F.col("count").desc()
).show(truncate=False)

=== Validation 2: Standardized Role Distribution ===


+----------+-----+
|role      |count|
+----------+-----+
|flex      |21   |
|duelist   |17   |
|controller|14   |
|support   |14   |
|sentinel  |13   |
|NULL      |1    |
+----------+-----+

=== Validation 2: Standardized Status Distribution ===
+----------+-----+
|status    |count|
+----------+-----+
|active    |42   |
|inactive  |21   |
|substitute|17   |
+----------+-----+



**validity**

In [37]:
df_players_standardized.agg(
    F.min("join_date").alias("earliest_date"),
    F.max("join_date").alias("latest_date"),
).show(truncate=False)

+-------------+-----------+
|earliest_date|latest_date|
+-------------+-----------+
|2023-01-31   |2026-03-30 |
+-------------+-----------+



In [38]:
# the dates are in a valid range

In [39]:
print("=== Player Status Counts ===")
df_players_standardized.groupBy("status").count().orderBy(
    F.col("count").desc()
).show(truncate=False)

=== Player Status Counts ===
+----------+-----+
|status    |count|
+----------+-----+
|active    |42   |
|inactive  |21   |
|substitute|17   |
+----------+-----+



**matches.csv**

In [40]:
# 1. Load matches.csv into a Spark DataFrame
df_matches = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_path}/matches.csv")

print("=== 1. Column Names and Data Types ===")
df_matches.printSchema()

print(f"\n=== 2. Total Number of Rows ===")
print(f"Total rows: {df_matches.count()}")

print(f"\n=== 3. Sample of Actual Records ===")
df_matches.show(5, truncate=False)

print(f"\n=== 4. Number of Distinct match_id Values ===")
distinct_matches = df_matches.select("match_id").distinct().count()
print(f"Distinct match_id count: {distinct_matches}")

=== 1. Column Names and Data Types ===
root
 |-- match_id: string (nullable = true)
 |-- match_time: string (nullable = true)
 |-- game: string (nullable = true)
 |-- team_a_id: string (nullable = true)
 |-- team_b_id: string (nullable = true)
 |-- team_a_score: string (nullable = true)
 |-- team_b_score: string (nullable = true)
 |-- match_status: string (nullable = true)


=== 2. Total Number of Rows ===
Total rows: 402

=== 3. Sample of Actual Records ===
+--------+-------------------+-----------------+---------+---------+------------+------------+------------+
|match_id|match_time         |game             |team_a_id|team_b_id|team_a_score|team_b_score|match_status|
+--------+-------------------+-----------------+---------+---------+------------+------------+------------+
|MT0001  |2026-09-10 23:30:00|League of Legends|TM05     |TM01     |2           |3           |cancelled   |
|MT0002  |2026-09-11 19:15:00|Rocket League    |TM05     |TM08     |0           |1           |completed  

**de duplication**

In [41]:
df_matches.groupBy("match_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy("match_id") \
    .show(truncate=False)

+--------+-----+
|match_id|count|
+--------+-----+
|MT0004  |2    |
|MT0022  |2    |
+--------+-----+



In [42]:
# inspecting the exact records
df_matches.filter(
    F.col("match_id").isin("MT0004", "MT0022")
).orderBy("match_id").show(truncate=False)

+--------+-------------------+-----------------+---------+---------+------------+------------+------------+
|match_id|match_time         |game             |team_a_id|team_b_id|team_a_score|team_b_score|match_status|
+--------+-------------------+-----------------+---------+---------+------------+------------+------------+
|MT0004  |2026-09-11 05:45:00|League of Legends|TM05     |TM01     |3           |0           |completed   |
|MT0004  |2026-09-11 05:45:00|League of Legends|TM05     |TM01     |3           |0           |completed   |
|MT0022  |2026-09-17 16:45:00|Overwatch        |TM05     |TM07     |2           |1           |forfeit     |
|MT0022  |2026-09-17 16:45:00|Overwatch        |TM05     |TM07     |2           |1           |forfeit     |
+--------+-------------------+-----------------+---------+---------+------------+------------+------------+



In [43]:
# I could see that the records are exact duplicates
# So, simply dropping one row per the duplicate would work

In [44]:
df_matches_cleaned = df_matches.dropDuplicates()

print("Original row count:", df_matches.count())
print("Cleaned row count:", df_matches_cleaned.count())

print("Distinct match IDs:", df_matches_cleaned.select("match_id").distinct().count())

Original row count: 402
Cleaned row count: 400
Distinct match IDs: 400


**NULL handling**

In [45]:
# Build expressions to count anomalies for each column in df_matches_cleaned
missing_checks_matches = []
columns_to_check_matches = [
    "match_id",
    "match_time",
    "game",
    "team_a_id",
    "team_b_id",
    "team_a_score",
    "team_b_score",
    "match_status",
]

for c in columns_to_check_matches:
  condition = (
      F.col(c).isNull()
      | (F.col(c) == "")
      | (F.length(F.trim(F.col(c))) == 0)
  )

  missing_checks_matches.append(
      F.sum(F.when(condition, 1).otherwise(0)).alias(f"{c}_missing_count")
  )

# Execute the check on df_matches_cleaned
df_matches_cleaned.select(missing_checks_matches).show(truncate=False)

+----------------------+------------------------+------------------+-----------------------+-----------------------+--------------------------+--------------------------+--------------------------+
|match_id_missing_count|match_time_missing_count|game_missing_count|team_a_id_missing_count|team_b_id_missing_count|team_a_score_missing_count|team_b_score_missing_count|match_status_missing_count|
+----------------------+------------------------+------------------+-----------------------+-----------------------+--------------------------+--------------------------+--------------------------+
|0                     |0                       |0                 |0                      |0                      |0                         |0                         |0                         |
+----------------------+------------------------+------------------+-----------------------+-----------------------+--------------------------+--------------------------+--------------------------+



**validating primary key**

In [46]:
# Check for any match_id values with a count greater than 1
df_matches_cleaned.groupBy("match_id").count().filter(
    F.col("count") > 1
).show(truncate=False)

+--------+-----+
|match_id|count|
+--------+-----+
+--------+-----+



**Dealing with timestamps**

In [47]:
# Use try_to_timestamp so invalid strings return NULL instead of throwing an exception
df_matches_cleaned.withColumn(
    "parsed_match_time", F.try_to_timestamp(F.col("match_time"))
).filter(F.col("match_time").isNotNull() & F.col("parsed_match_time").isNull()).select(
    "match_id", "match_time", "match_status"
).show(
    truncate=False
)

+--------+----------+------------+
|match_id|match_time|match_status|
+--------+----------+------------+
|MT0032  |TBD       |Completed   |
+--------+----------+------------+



In [48]:
# Create df_matches_processed with parsed timestamps, a DQ flag, and standardized status casing
# for handling the match_time column values which cant be parsed
df_matches_processed = df_matches_cleaned.withColumn(
    "parsed_match_time", F.try_to_timestamp(F.col("match_time"))
).withColumn(
    "dq_status",
    F.when(F.col("parsed_match_time").isNotNull(), "VALID")
    .when(F.col("match_time") == "TBD", "FLAGGED_TBD_TIMESTAMP")
    .otherwise("FLAGGED_INVALID_TIMESTAMP"),
).withColumn(
    "match_status", F.lower(F.col("match_status"))
)

In [49]:
# Verify how match MT0032 looks with our new columns
df_matches_processed.filter(F.col("match_id") == "MT0032").select(
    "match_id",
    "match_time",
    "parsed_match_time",
    "match_status",
    "dq_status",
).show(truncate=False)

+--------+----------+-----------------+------------+---------------------+
|match_id|match_time|parsed_match_time|match_status|dq_status            |
+--------+----------+-----------------+------------+---------------------+
|MT0032  |TBD       |NULL             |completed   |FLAGGED_TBD_TIMESTAMP|
+--------+----------+-----------------+------------+---------------------+



In [50]:
df_matches_processed.show()

+--------+-------------------+-----------------+---------+---------+------------+------------+------------+-------------------+---------+
|match_id|         match_time|             game|team_a_id|team_b_id|team_a_score|team_b_score|match_status|  parsed_match_time|dq_status|
+--------+-------------------+-----------------+---------+---------+------------+------------+------------+-------------------+---------+
|  MT0122|2026-09-13 01:15:00|    Rocket League|     TM07|     TM04|           2|           0|   completed|2026-09-13 01:15:00|    VALID|
|  MT0206|2026-09-10 02:30:00|League of Legends|     TM01|     TM04|           3|           0|   completed|2026-09-10 02:30:00|    VALID|
|  MT0276|2026-09-01 10:45:00|         Valorant|     TM05|     TM02|           0|           2|   completed|2026-09-01 10:45:00|    VALID|
|  MT0011|2026-09-07 05:45:00|League of Legends|     TM06|     TM04|           0|           1|   cancelled|2026-09-07 05:45:00|    VALID|
|  MT0252|2026-09-05 11:45:00|    

**Score verification**

In [51]:
df_matches_cleaned.select(
    "match_id",
    "game",
    "team_a_score",
    "team_b_score",
    "match_status"
).show(20, truncate=False)

+--------+-----------------+------------+------------+------------+
|match_id|game             |team_a_score|team_b_score|match_status|
+--------+-----------------+------------+------------+------------+
|MT0122  |Rocket League    |2           |0           |Completed   |
|MT0206  |League of Legends|3           |0           |completed   |
|MT0276  |Valorant         |0           |2           |completed   |
|MT0011  |League of Legends|0           |1           |cancelled   |
|MT0252  |Valorant         |3           |3           |cancelled   |
|MT0102  |Overwatch        |0           |2           |completed   |
|MT0172  |Overwatch        |0           |2           |completed   |
|MT0261  |Rocket League    |1           |3           |cancelled   |
|MT0340  |Valorant         |3           |1           |completed   |
|MT0287  |Rocket League    |0           |0           |completed   |
|MT0111  |Rocket League    |1           |0           |completed   |
|MT0142  |League of Legends|1           |1      

In [52]:
# finding invalid scores
for c in ["team_a_score", "team_b_score"]:
    print(f"Checking {c}")

    df_matches_cleaned.groupBy(c) \
        .count() \
        .orderBy(F.desc("count")) \
        .show(truncate=False)

Checking team_a_score
+------------+-----+
|team_a_score|count|
+------------+-----+
|1           |103  |
|2           |101  |
|3           |96   |
|0           |90   |
|unknown     |4    |
|N/A         |3    |
|-           |2    |
|??          |1    |
+------------+-----+

Checking team_b_score
+------------+-----+
|team_b_score|count|
+------------+-----+
|2           |104  |
|0           |101  |
|1           |95   |
|3           |93   |
|-           |4    |
|??          |2    |
|unknown     |1    |
+------------+-----+



In [53]:
# List of recognized placeholder strings we want to map to NULL
placeholders = ["unknown", "N/A", "-", "??"]

# Create cleaned score columns and data quality score flags
df_matches_processed = (
    df_matches_processed
    # For Team A Score
    .withColumn(
        "clean_team_a_score",
        F.when(F.col("team_a_score").isin("0", "1", "2", "3"), F.col("team_a_score").cast("int"))
        .otherwise(None)
    )
    .withColumn(
        "team_a_score_flag",
        F.when(F.col("team_a_score").isin("0", "1", "2", "3"), "VALID")
        .otherwise("FLAGGED_INVALID_SCORE")
    )
    # For Team B Score
    .withColumn(
        "clean_team_b_score",
        F.when(F.col("team_b_score").isin("0", "1", "2", "3"), F.col("team_b_score").cast("int"))
        .otherwise(None)
    )
    .withColumn(
        "team_b_score_flag",
        F.when(F.col("team_b_score").isin("0", "1", "2", "3"), "VALID")
        .otherwise("FLAGGED_INVALID_SCORE")
    )
)

# Inspect a few flagged rows (e.g. where team_a_score had '??' or 'unknown')
df_matches_processed.filter(
    (F.col("team_a_score_flag") == "FLAGGED_INVALID_SCORE") |
    (F.col("team_b_score_flag") == "FLAGGED_INVALID_SCORE")
).select(
    "match_id", "team_a_score", "clean_team_a_score", "team_a_score_flag",
    "team_b_score", "clean_team_b_score", "team_b_score_flag"
).show(10, truncate=False)

+--------+------------+------------------+---------------------+------------+------------------+---------------------+
|match_id|team_a_score|clean_team_a_score|team_a_score_flag    |team_b_score|clean_team_b_score|team_b_score_flag    |
+--------+------------+------------------+---------------------+------------+------------------+---------------------+
|MT0026  |??          |NULL              |FLAGGED_INVALID_SCORE|3           |3                 |VALID                |
|MT0151  |0           |0                 |VALID                |-           |NULL              |FLAGGED_INVALID_SCORE|
|MT0189  |N/A         |NULL              |FLAGGED_INVALID_SCORE|3           |3                 |VALID                |
|MT0078  |2           |2                 |VALID                |unknown     |NULL              |FLAGGED_INVALID_SCORE|
|MT0270  |-           |NULL              |FLAGGED_INVALID_SCORE|0           |0                 |VALID                |
|MT0082  |2           |2                 |VALID 

In [54]:
df_matches_processed.select(
    F.sum(
        F.when(
            F.col("team_a_score_flag") == "FLAGGED_INVALID_SCORE", 1
        ).otherwise(0)
    ).alias("invalid_team_a_scores"),

    F.sum(
        F.when(
            F.col("team_b_score_flag") == "FLAGGED_INVALID_SCORE", 1
        ).otherwise(0)
    ).alias("invalid_team_b_scores"),

    F.count("*").alias("total_rows")
).show()

+---------------------+---------------------+----------+
|invalid_team_a_scores|invalid_team_b_scores|total_rows|
+---------------------+---------------------+----------+
|                   10|                    7|       400|
+---------------------+---------------------+----------+



**standardization**

In [55]:
df_matches_processed.groupBy("match_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+------------+-----+
|match_status|count|
+------------+-----+
|completed   |253  |
|cancelled   |81   |
|forfeit     |66   |
+------------+-----+



**checking with both are same teams**

In [56]:
df_matches_processed.filter(
    F.col("team_a_id") == F.col("team_b_id")
).select(
    "match_id",
    "game",
    "team_a_id",
    "team_b_id",
    "match_status"
).show(truncate=False)

+--------+-----------------+---------+---------+------------+
|match_id|game             |team_a_id|team_b_id|match_status|
+--------+-----------------+---------+---------+------------+
|MT0013  |League of Legends|TM08     |TM08     |forfeit     |
+--------+-----------------+---------+---------+------------+



In [57]:
# want to place this in quarantine dataset
# becuase a team playing with itself doesnt make sense
# So, it needs further inspection

In [58]:
df_matches_processed.groupBy("game") \
    .count() \
    .orderBy("game") \
    .show(truncate=False)

+-----------------+-----+
|game             |count|
+-----------------+-----+
|League of Legends|105  |
|Overwatch        |92   |
|Rocket League    |93   |
|Valorant         |110  |
+-----------------+-----+



**player_match_stats.csv**

In [59]:
# Load the CSV if you haven't already
player_stats_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{data_path}/player_match_stats.csv")

# Inspect the structure
player_stats_df.printSchema()

# Check the number of rows
print("Total rows:", player_stats_df.count())

# Inspect sample records
player_stats_df.show(10, truncate=False)

root
 |-- stat_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- kills: string (nullable = true)
 |-- deaths: string (nullable = true)
 |-- assists: string (nullable = true)
 |-- damage: string (nullable = true)
 |-- rounds_played: string (nullable = true)

Total rows: 1803
+-------+--------+---------+-----+------+-------+------+-------------+
|stat_id|match_id|player_id|kills|deaths|assists|damage|rounds_played|
+-------+--------+---------+-----+------+-------+------+-------------+
|ST00001|MT0035  |PL077    |29   |6     |21     |10178 |13           |
|ST00002|MT0396  |PL002    |16   |11    |24     |5902  |unknown      |
|ST00003|MT0252  |PL026    |33   |20    |24     |6303  |13           |
|ST00004|MT0062  |PL071    |30   |22    |12     |5641  |12           |
|ST00005|MT0013  |PL022    |8    |0     |22     |7568  |NULL         |
|ST00006|MT0215  |PL075    |23   |3     |??     |8112  |13           |
|ST00007|MT0248  |PL0

**primary key**

In [60]:
print("Total rows:", player_stats_df.count())

print(
    "Distinct stat IDs:",
    player_stats_df.select("stat_id").distinct().count()
)

player_stats_df.groupBy("stat_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

Total rows: 1803
Distinct stat IDs: 1800
+-------+-----+
|stat_id|count|
+-------+-----+
|ST00009|2    |
|ST00100|2    |
|ST00202|2    |
+-------+-----+



In [61]:
# investigating all duplicate records
duplicate_stat_ids = ["ST00009", "ST00100", "ST00202"]

player_stats_df.filter(
    F.col("stat_id").isin(duplicate_stat_ids)
).orderBy("stat_id").show(truncate=False)

+-------+--------+---------+-----+------+-------+------+-------------+
|stat_id|match_id|player_id|kills|deaths|assists|damage|rounds_played|
+-------+--------+---------+-----+------+-------+------+-------------+
|ST00009|MT0058  |PL001    |12   |27    |17     |5576  |20           |
|ST00009|MT0058  |PL001    |12   |27    |17     |5576  |20           |
|ST00100|MT0178  |PL048    |13   |8     |3      |11395 |24           |
|ST00100|MT0178  |PL048    |13   |8     |3      |11395 |24           |
|ST00202|MT0279  |PL035    |11   |25    |3      |9957  |13           |
|ST00202|MT0279  |PL035    |11   |25    |3      |9957  |13           |
+-------+--------+---------+-----+------+-------+------+-------------+



In [62]:
# These are exact same values
# so, just dropping the duplicates would work

In [63]:
player_stats_clean = player_stats_df.dropDuplicates()

print("Original rows:", player_stats_df.count())
print("Cleaned rows:", player_stats_clean.count())

print(
    "Distinct stat IDs:",
    player_stats_clean.select("stat_id").distinct().count()
)

Original rows: 1803
Cleaned rows: 1800
Distinct stat IDs: 1800


**NULL handling**

In [64]:
columns_to_check = player_stats_clean.columns

missing_checks = []

for c in columns_to_check:
    condition = (
        F.col(c).isNull()
        | (F.length(F.trim(F.col(c))) == 0)
    )

    missing_checks.append(
        F.sum(
            F.when(condition, 1).otherwise(0)
        ).alias(f"{c}_missing_count")
    )

player_stats_clean.select(missing_checks).show(truncate=False)

+---------------------+----------------------+-----------------------+-------------------+--------------------+---------------------+--------------------+---------------------------+
|stat_id_missing_count|match_id_missing_count|player_id_missing_count|kills_missing_count|deaths_missing_count|assists_missing_count|damage_missing_count|rounds_played_missing_count|
+---------------------+----------------------+-----------------------+-------------------+--------------------+---------------------+--------------------+---------------------------+
|0                    |0                     |0                      |0                  |0                   |0                    |0                   |293                        |
+---------------------+----------------------+-----------------------+-------------------+--------------------+---------------------+--------------------+---------------------------+



In [65]:
# investigating the rounds_played column
player_stats_clean.groupBy("rounds_played") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+-------------+-----+
|rounds_played|count|
+-------------+-----+
|20           |312  |
|unknown      |304  |
|24           |301  |
|12           |297  |
|NULL         |293  |
|13           |293  |
+-------------+-----+



In [66]:
numeric_columns = [
    "kills",
    "deaths",
    "assists",
    "damage",
    "rounds_played"
]

for c in numeric_columns:
    print(f"--- {c} ---")

    player_stats_clean.groupBy(c) \
        .count() \
        .orderBy(F.desc("count")) \
        .show(truncate=False)

--- kills ---
+-----+-----+
|kills|count|
+-----+-----+
|3    |67   |
|33   |62   |
|35   |61   |
|27   |61   |
|32   |61   |
|31   |60   |
|6    |58   |
|21   |55   |
|10   |54   |
|7    |53   |
|8    |52   |
|9    |52   |
|2    |52   |
|34   |50   |
|17   |50   |
|24   |50   |
|19   |49   |
|1    |49   |
|16   |48   |
|14   |48   |
+-----+-----+
only showing top 20 rows
--- deaths ---
+------+-----+
|deaths|count|
+------+-----+
|16    |79   |
|11    |71   |
|30    |68   |
|24    |66   |
|25    |64   |
|14    |64   |
|8     |62   |
|2     |62   |
|19    |61   |
|6     |60   |
|7     |58   |
|3     |57   |
|27    |57   |
|20    |57   |
|17    |56   |
|23    |56   |
|9     |56   |
|4     |56   |
|12    |55   |
|5     |54   |
+------+-----+
only showing top 20 rows
--- assists ---
+-------+-----+
|assists|count|
+-------+-----+
|23     |81   |
|13     |77   |
|19     |75   |
|17     |74   |
|10     |73   |
|4      |73   |
|14     |73   |
|16     |72   |
|24     |72   |
|11     |71   |
|

In [67]:
# checking the non numeric values
numeric_columns = [
    "kills",
    "deaths",
    "assists",
    "damage",
    "rounds_played"
]

for c in numeric_columns:
    print(f"--- Non-numeric values in {c} ---")

    player_stats_clean.filter(
        F.col(c).isNotNull() &
        ~F.trim(F.col(c)).rlike(r"^[0-9]+$")
    ).groupBy(c).count().orderBy(
        F.desc("count")
    ).show(truncate=False)

--- Non-numeric values in kills ---
+-------+-----+
|kills  |count|
+-------+-----+
|N/A    |12   |
|-      |6    |
|??     |6    |
|unknown|5    |
|-4     |1    |
+-------+-----+

--- Non-numeric values in deaths ---
+-------+-----+
|deaths |count|
+-------+-----+
|unknown|13   |
|-      |13   |
|N/A    |6    |
|??     |6    |
+-------+-----+

--- Non-numeric values in assists ---
+-------+-----+
|assists|count|
+-------+-----+
|unknown|15   |
|N/A    |15   |
|??     |11   |
|-      |10   |
+-------+-----+

--- Non-numeric values in damage ---
+-------+-----+
|damage |count|
+-------+-----+
|-      |9    |
|unknown|6    |
|N/A    |5    |
|??     |4    |
|lots   |1    |
+-------+-----+

--- Non-numeric values in rounds_played ---
+-------------+-----+
|rounds_played|count|
+-------------+-----+
|unknown      |304  |
+-------------+-----+



In [68]:
# checking the min and max values of each col
numeric_columns = [
    "kills",
    "deaths",
    "assists",
    "damage",
    "rounds_played"
]

for c in numeric_columns:
    parsed_col = F.expr(f"try_cast(trim(`{c}`) AS INT)")

    print(f"--- Range check: {c} ---")

    player_stats_clean.select(
        parsed_col.alias("parsed_value")
    ).agg(
        F.min("parsed_value").alias("min_value"),
        F.max("parsed_value").alias("max_value"),
        F.sum(
            F.when(F.col("parsed_value").isNull(), 1).otherwise(0)
        ).alias("unparseable_or_missing")
    ).show()

--- Range check: kills ---
+---------+---------+----------------------+
|min_value|max_value|unparseable_or_missing|
+---------+---------+----------------------+
|       -4|       35|                    29|
+---------+---------+----------------------+

--- Range check: deaths ---
+---------+---------+----------------------+
|min_value|max_value|unparseable_or_missing|
+---------+---------+----------------------+
|        0|       30|                    38|
+---------+---------+----------------------+

--- Range check: assists ---
+---------+---------+----------------------+
|min_value|max_value|unparseable_or_missing|
+---------+---------+----------------------+
|        0|       25|                    51|
+---------+---------+----------------------+

--- Range check: damage ---
+---------+---------+----------------------+
|min_value|max_value|unparseable_or_missing|
+---------+---------+----------------------+
|      102|    11993|                    25|
+---------+---------+---------

In [69]:
# Create df_player_stats_processed to handle the kills column logic
df_player_stats_processed = (
    player_stats_clean
    # 1. Preserve original kills value in the raw form (it's already in the dataframe)
    # 2 & 3. Parse to integer, but invalidate if it's negative or unparseable
    .withColumn(
        "parsed_kills",
        F.expr("try_cast(trim(kills) AS INT)")
    )
    .withColumn(
        "clean_kills",
        F.when(
            (F.col("parsed_kills").isNotNull()) & (F.col("parsed_kills") >= 0),
            F.col("parsed_kills")
        ).otherwise(None)
    )
    # 4. Add a specific DQ status flag for tracking
    .withColumn(
        "kills_dq_status",
        F.when(F.col("parsed_kills").isNull(), "FLAGGED_UNPARSEABLE_KILLS")
        .when(F.col("parsed_kills") < 0, "FLAGGED_NEGATIVE_KILLS")
        .otherwise("VALID")
    )
)

# 5. Inspect the row with the negative kill count to verify our logic worked
df_player_stats_processed.filter(F.col("kills_dq_status") == "FLAGGED_NEGATIVE_KILLS").select(
    "stat_id", "match_id", "player_id", "kills", "clean_kills", "kills_dq_status"
).show(truncate=False)

+-------+--------+---------+-----+-----------+----------------------+
|stat_id|match_id|player_id|kills|clean_kills|kills_dq_status       |
+-------+--------+---------+-----+-----------+----------------------+
|ST00020|MT0091  |PL070    |-4   |NULL       |FLAGGED_NEGATIVE_KILLS|
+-------+--------+---------+-----+-----------+----------------------+



**Uniqueness of business key**

In [70]:
# here for checking the uniqueness
# I am assuming that for a single match, each player can have only a single stat
# so I am taking a combination of player_id and match_id to check it

In [71]:
# Check for duplicate combinations of match_id and player_id on processed data
df_duplicate_player_matches = (
    df_player_stats_processed.groupBy("match_id", "player_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicate player-match combinations found:"
    f" {df_duplicate_player_matches.count()}"
)
df_duplicate_player_matches.show(10, truncate=False)

Duplicate player-match combinations found: 52
+--------+---------+-----+
|match_id|player_id|count|
+--------+---------+-----+
|MT0072  |PL012    |2    |
|MT0147  |PL031    |2    |
|MT0044  |PL040    |2    |
|MT0284  |PL018    |2    |
|MT0229  |PL055    |2    |
|MT0195  |PL034    |2    |
|MT0032  |PL043    |2    |
|MT0169  |PL006    |2    |
|MT0010  |PL045    |2    |
|MT0332  |PL011    |2    |
+--------+---------+-----+
only showing top 10 rows


In [72]:
# Join back or filter df_player_stats_processed to show the actual duplicate rows
df_player_stats_processed.join(
    df_duplicate_player_matches.select("match_id", "player_id"),
    ["match_id", "player_id"],
    "inner"
).orderBy("match_id", "player_id").show(20, truncate=False)

+--------+---------+-------+-----+------+-------+------+-------------+------------+-----------+---------------+
|match_id|player_id|stat_id|kills|deaths|assists|damage|rounds_played|parsed_kills|clean_kills|kills_dq_status|
+--------+---------+-------+-----+------+-------+------+-------------+------------+-----------+---------------+
|MT0002  |PL024    |ST01203|18   |2     |10     |4526  |24           |18          |18         |VALID          |
|MT0002  |PL024    |ST00816|15   |12    |15     |3736  |NULL         |15          |15         |VALID          |
|MT0009  |PL036    |ST01461|6    |2     |9      |3734  |unknown      |6           |6          |VALID          |
|MT0009  |PL036    |ST00696|15   |17    |8      |8545  |20           |15          |15         |VALID          |
|MT0010  |PL045    |ST00607|11   |11    |14     |8459  |20           |11          |11         |VALID          |
|MT0010  |PL045    |ST00919|17   |7     |2      |220   |20           |17          |17         |VALID    

In [73]:
# These are players which are having multiple stats in a single match
# which actually violates our assumption that one player can have only one stat in a match
# So,I decided to quarantine these records

In [74]:
# Identify all records belonging to repeated player-match combinations
df_player_stats_quarantine = (
    df_player_stats_processed.join(
        df_duplicate_player_matches.select("match_id", "player_id"),
        on=["match_id", "player_id"],
        how="inner"
    )
)

# Keep records whose player-match combination is not duplicated
df_player_stats_valid = (
    df_player_stats_processed.join(
        df_duplicate_player_matches.select("match_id", "player_id"),
        on=["match_id", "player_id"],
        how="left_anti"
    )
)

In [75]:
#validating the results
print("Original processed rows:", df_player_stats_processed.count())
print("Quarantined rows:", df_player_stats_quarantine.count())
print("Valid rows:", df_player_stats_valid.count())

print(
    "Quarantined distinct player-match combinations:",
    df_player_stats_quarantine
        .select("match_id", "player_id")
        .distinct()
        .count()
)

print(
    "Valid duplicate player-match combinations:",
    df_player_stats_valid
        .groupBy("match_id", "player_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

Original processed rows: 1800
Quarantined rows: 105
Valid rows: 1695
Quarantined distinct player-match combinations: 52
Valid duplicate player-match combinations: 0


**match_events.csv**

In [76]:
# 1. Load match_events.csv into a Spark DataFrame
df_match_events = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_path}/match_events.csv")

print("=== 1. Column Names and Data Types ===")
df_match_events.printSchema()

print(f"\n=== 2. Total Number of Rows ===")
print(f"Total rows: {df_match_events.count()}")

print(f"\n=== 3. Sample of Actual Records ===")
df_match_events.show(5, truncate=False)

=== 1. Column Names and Data Types ===
root
 |-- event_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- duration_seconds: string (nullable = true)


=== 2. Total Number of Rows ===
Total rows: 1000

=== 3. Sample of Actual Records ===
+--------+--------+-------------------+---------------+----------------+
|event_id|match_id|event_time         |event_type     |duration_seconds|
+--------+--------+-------------------+---------------+----------------+
|EV00001 |MT0310  |2026-09-16 14:02:00|substitution   |402             |
|EV00002 |MT0107  |2026-09-20 23:09:00|substitution   |196             |
|EV00003 |MT0148  |2026-09-07 11:07:00|technical_pause|755             |
|EV00004 |MT0355  |2026-09-11 09:18:00|technical_pause|187             |
|EV00005 |MT0125  |2026-09-17 22:25:00|technical_pause|528             |
+--------+--------+-------------------+---------------+----------------+

**duplicate handling**

In [77]:
# Check exact duplicate rows
total_rows = df_match_events.count()
distinct_rows = df_match_events.distinct().count()

print("Total rows:", total_rows)
print("Distinct full rows:", distinct_rows)
print("Exact duplicate rows:", total_rows - distinct_rows)

# Check duplicate event IDs
df_duplicate_event_ids = (
    df_match_events
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate event IDs:",
    df_duplicate_event_ids.count()
)

df_duplicate_event_ids.show(truncate=False)

Total rows: 1000
Distinct full rows: 1000
Exact duplicate rows: 0
Duplicate event IDs: 0
+--------+-----+
|event_id|count|
+--------+-----+
+--------+-----+



**Missing values**

In [78]:
missing_checks = []

for c in df_match_events.columns:
    missing_checks.append(
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
    )

df_match_events.select(*missing_checks).show(
    truncate=False,
    vertical=True
)

-RECORD 0---------------
 event_id         | 0   
 match_id         | 0   
 event_time       | 0   
 event_type       | 0   
 duration_seconds | 0   



In [79]:
# converting duration_seconds into integer and converting
# incompatible values into NULL
# Create df_match_events_processed to handle duration_seconds safely
df_match_events_processed = (
    df_match_events
    # Try casting the trimmed string to an integer
    .withColumn(
        "parsed_duration",
        F.expr("try_cast(trim(duration_seconds) AS INT)")
    )
    # Clean duration: must be non-null and non-negative
    .withColumn(
        "clean_duration_seconds",
        F.when(
            (F.col("parsed_duration").isNotNull()) & (F.col("parsed_duration") >= 0),
            F.col("parsed_duration")
        ).otherwise(None)
    )
    # Add a Data Quality (DQ) flag column for traceability
    .withColumn(
        "duration_dq_status",
        F.when(F.col("parsed_duration").isNull(), "FLAGGED_UNPARSEABLE_DURATION")
        .when(F.col("parsed_duration") < 0, "FLAGGED_NEGATIVE_DURATION")
        .otherwise("VALID")
    )
)

# Inspect the distribution of our new DQ status flag to see how many events are valid vs flagged
df_match_events_processed.groupBy("duration_dq_status").count().show(truncate=False)

+----------------------------+-----+
|duration_dq_status          |count|
+----------------------------+-----+
|VALID                       |976  |
|FLAGGED_UNPARSEABLE_DURATION|24   |
+----------------------------+-----+



In [80]:
df_match_events_processed.filter(
    F.col("parsed_duration") < 0
).select(
    "event_id",
    "match_id",
    "duration_seconds",
    "parsed_duration",
    "clean_duration_seconds",
    "duration_dq_status"
).show(truncate=False)

+--------+--------+----------------+---------------+----------------------+------------------+
|event_id|match_id|duration_seconds|parsed_duration|clean_duration_seconds|duration_dq_status|
+--------+--------+----------------+---------------+----------------------+------------------+
+--------+--------+----------------+---------------+----------------------+------------------+



In [81]:
df_match_events_processed.filter(
    F.col("duration_dq_status") == "FLAGGED_UNPARSEABLE_DURATION"
).groupBy(
    "duration_seconds"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

+----------------+-----+
|duration_seconds|count|
+----------------+-----+
|unknown         |9    |
|N/A             |6    |
|-               |5    |
|??              |4    |
+----------------+-----+



In [82]:
df_match_events_processed.groupBy(
    "event_type"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

+---------------+-----+
|event_type     |count|
+---------------+-----+
|technical_pause|184  |
|substitution   |174  |
|disconnect     |168  |
|highlight      |164  |
|timeout        |158  |
|penalty        |152  |
+---------------+-----+



In [83]:
# Use try_to_timestamp without an extra format string argument
df_match_events_processed = df_match_events_processed.withColumn(
    "parsed_event_time", F.try_to_timestamp(F.trim(F.col("event_time")))
)

In [84]:
df_match_events_processed.groupBy(
    F.when(
        F.col("parsed_event_time").isNull(),
        "INVALID_OR_MISSING"
    ).otherwise("VALID").alias("event_time_status")
).count().show()

+------------------+-----+
| event_time_status|count|
+------------------+-----+
|             VALID|  999|
|INVALID_OR_MISSING|    1|
+------------------+-----+



In [85]:
df_match_events_processed.filter(
    F.col("parsed_event_time").isNull()
).select(
    "event_id",
    "match_id",
    "event_time",
    "event_type",
    "duration_seconds"
).show(truncate=False)

+--------+--------+----------+----------+----------------+
|event_id|match_id|event_time|event_type|duration_seconds|
+--------+--------+----------+----------+----------------+
|EV00010 |MT0376  |unknown   |disconnect|486             |
+--------+--------+----------+----------+----------------+



In [86]:
# adding new column to track these records
df_match_events_processed = (
    df_match_events_processed
    .withColumn(
        "event_time_dq_status",
        F.when(
            F.col("parsed_event_time").isNull(),
            "FLAGGED_INVALID_EVENT_TIME"
        ).otherwise("VALID")
    )
)

In [87]:
df_match_events_processed.groupBy(
    "event_time_dq_status"
).count().show(truncate=False)

+--------------------------+-----+
|event_time_dq_status      |count|
+--------------------------+-----+
|VALID                     |999  |
|FLAGGED_INVALID_EVENT_TIME|1    |
+--------------------------+-----+



In [88]:
columns = df_match_events.columns

df_match_events.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == ""),
            1
        )
    ).alias(c)
    for c in columns
]).show()

+--------+--------+----------+----------+----------------+
|event_id|match_id|event_time|event_type|duration_seconds|
+--------+--------+----------+----------+----------------+
|       0|       0|         0|         0|               0|
+--------+--------+----------+----------+----------------+



**CROSS FILE VALIDATION**

**players.team_id → teams.team_id**

In [89]:
# finding all the players whose teams are not listed in the teams,csv
df_unmatched_players = df_players.join(
    df_teams.select("team_id"),
    on="team_id",
    how="left_anti"
)

In [91]:
df_unmatched_players.show()

+-------+---------+-----------+-------+----------+--------+
|team_id|player_id|player_name|   role| join_date|  status|
+-------+---------+-----------+-------+----------+--------+
|   TM99|    PL006|    Ghost_6|support|2023-07-15|inactive|
+-------+---------+-----------+-------+----------+--------+



In [92]:
# I could see that this is player is currently inactive
# Also He is from a team that is not present in our teams list
# so he might be an old player whose team is currently not presnt
# So, I want to avoid using him in my analysis
# will quarantine this record for further investigation

In [93]:
# Create df_players_quarantine for players whose team_id is missing from teams reference
df_players_quarantine = df_players.join(
    df_teams.select("team_id"),
    on="team_id",
    how="left_anti"
)

# Create df_players_valid containing only players belonging to valid teams
df_players_valid = df_players.join(
    df_teams.select("team_id"),
    on="team_id",
    how="left_semi"
)

# Validate the counts
print(f"Original players count: {df_players.count()}")
print(f"Quarantined players count: {df_players_quarantine.count()}")
print(f"Valid players count: {df_players_valid.count()}")

# Inspect the quarantined record
df_players_quarantine.show(truncate=False)

Original players count: 80
Quarantined players count: 1
Valid players count: 79
+-------+---------+-----------+-------+----------+--------+
|team_id|player_id|player_name|role   |join_date |status  |
+-------+---------+-----------+-------+----------+--------+
|TM99   |PL006    |Ghost_6    |support|2023-07-15|inactive|
+-------+---------+-----------+-------+----------+--------+



**matches.team_a_id → teams.team_id**

In [94]:
# Check if team_a_id in df_matches_processed exists in df_teams
df_invalid_team_a_matches = df_matches_processed.join(
    df_teams.select("team_id"),
    df_matches_processed["team_a_id"] == df_teams["team_id"],
    how="left_anti"
)

print(f"Matches with invalid/missing team_a_id: {df_invalid_team_a_matches.count()}")
df_invalid_team_a_matches.select("match_id", "team_a_id", "team_b_id", "game", "match_status").show(truncate=False)

Matches with invalid/missing team_a_id: 0
+--------+---------+---------+----+------------+
|match_id|team_a_id|team_b_id|game|match_status|
+--------+---------+---------+----+------------+
+--------+---------+---------+----+------------+



In [95]:
# Check if team_b_id in df_matches_processed exists in df_teams
df_invalid_team_b_matches = df_matches_processed.join(
    df_teams.select("team_id"),
    df_matches_processed["team_b_id"] == df_teams["team_id"],
    how="left_anti"
)

print(f"Matches with invalid/missing team_b_id: {df_invalid_team_b_matches.count()}")
df_invalid_team_b_matches.select("match_id", "team_a_id", "team_b_id", "game", "match_status").show(truncate=False)

Matches with invalid/missing team_b_id: 0
+--------+---------+---------+----+------------+
|match_id|team_a_id|team_b_id|game|match_status|
+--------+---------+---------+----+------------+
+--------+---------+---------+----+------------+



**Player match stats → Players**

In [96]:
# Check if player_id in df_player_stats_valid exists in df_players_valid
df_invalid_player_stats = df_player_stats_valid.join(
    df_players_valid.select("player_id"),
    on="player_id",
    how="left_anti"
)

print(f"Stats records referencing missing/invalid players: {df_invalid_player_stats.count()}")
df_invalid_player_stats.select("stat_id", "match_id", "player_id", "kills").show(truncate=False)

Stats records referencing missing/invalid players: 20
+-------+--------+---------+-------+
|stat_id|match_id|player_id|kills  |
+-------+--------+---------+-------+
|ST01346|MT0356  |PL006    |31     |
|ST01440|MT0306  |PL006    |3      |
|ST01222|MT0098  |PL006    |1      |
|ST00701|MT0055  |PL006    |29     |
|ST00931|MT0270  |PL006    |2      |
|ST00883|MT0304  |PL006    |7      |
|ST00840|MT0263  |PL006    |N/A    |
|ST01734|MT0225  |PL006    |24     |
|ST01587|MT0180  |PL006    |25     |
|ST00011|MT0091  |PL999    |8      |
|ST00488|MT0275  |PL006    |34     |
|ST00500|MT0134  |PL006    |3      |
|ST01374|MT0146  |PL006    |15     |
|ST00469|MT0007  |PL006    |22     |
|ST00113|MT0059  |PL006    |23     |
|ST01567|MT0206  |PL006    |unknown|
|ST01347|MT0078  |PL006    |24     |
|ST01140|MT0318  |PL006    |21     |
|ST01667|MT0104  |PL006    |12     |
|ST00322|MT0014  |PL006    |21     |
+-------+--------+---------+-------+



In [97]:
# Here, previously we quarantined PL006 as his team was not present 
# and he was inactive
# Now, I got these stats of him
# And there is one more player PL999 who is not in the players.csv

In [98]:
# So, I decided to flag both of them and quarantine them
# for further investigation

In [99]:
from pyspark.sql import functions as F

# Combine our check to capture both PL006 (orphaned team) and PL999 (ghost player)
df_invalid_player_stats_flagged = df_player_stats_valid.join(
    df_players.select("player_id"),
    on="player_id",
    how="left_anti"
).withColumn(
    "stat_quarantine_reason",
    F.when(F.col("player_id") == "PL006", "ORPHANED_TEAM_PLAYER")
    .otherwise("UNKNOWN_GHOST_PLAYER")
)

# Inspect the breakdown of these invalid stats records
df_invalid_player_stats_flagged.groupBy("player_id", "stat_quarantine_reason").count().show(truncate=False)

+---------+----------------------+-----+
|player_id|stat_quarantine_reason|count|
+---------+----------------------+-----+
|PL999    |UNKNOWN_GHOST_PLAYER  |1    |
+---------+----------------------+-----+



**Player Match Stats → Matches.**

In [100]:
# Check if match_id in player stats exists in our processed matches reference dataset
df_invalid_match_stats = df_player_stats_valid.join(
    df_matches_processed.select("match_id"),
    on="match_id",
    how="left_anti"
)

print(f"Stats records referencing missing/invalid matches: {df_invalid_match_stats.count()}")
df_invalid_match_stats.select("stat_id", "match_id", "player_id", "kills").show(truncate=False)

Stats records referencing missing/invalid matches: 0
+-------+--------+---------+-----+
|stat_id|match_id|player_id|kills|
+-------+--------+---------+-----+
+-------+--------+---------+-----+



**Match Events → Matches**

In [101]:
# Check if match_id in df_match_events exists in our processed matches reference dataset
df_invalid_match_events = df_match_events.join(
    df_matches_processed.select("match_id"),
    on="match_id",
    how="left_anti"
)

print(f"Match events referencing missing/invalid matches: {df_invalid_match_events.count()}")
df_invalid_match_events.select("event_id", "match_id", "event_time", "event_type").show(truncate=False)

Match events referencing missing/invalid matches: 1
+--------+--------+-------------------+----------+
|event_id|match_id|event_time         |event_type|
+--------+--------+-------------------+----------+
|EV00006 |MT9999  |2026-09-14 17:48:00|disconnect|
+--------+--------+-------------------+----------+



In [102]:
# quarantining this event as it does not belong to  any matches present
# 1. Quarantine match events referencing missing/invalid match IDs
df_match_events_quarantine = df_match_events.join(
    df_matches_processed.select("match_id"),
    on="match_id",
    how="left_anti"
).withColumn("event_quarantine_reason", F.lit("ORPHANED_MATCH_EVENT"))

In [103]:
# 2. Keep only valid match events for downstream analysis
df_match_events_valid = df_match_events.join(
    df_matches_processed.select("match_id"),
    on="match_id",
    how="left_semi"
)

In [104]:
# Verify counts
print(f"Total match events: {df_match_events.count()}")
print(f"Quarantined match events: {df_match_events_quarantine.count()}")
print(f"Valid match events: {df_match_events_valid.count()}")

# Inspect the quarantined record
df_match_events_quarantine.show(truncate=False)

Total match events: 1000
Quarantined match events: 1
Valid match events: 999
+--------+--------+-------------------+----------+----------------+-----------------------+
|match_id|event_id|event_time         |event_type|duration_seconds|event_quarantine_reason|
+--------+--------+-------------------+----------+----------------+-----------------------+
|MT9999  |EV00006 |2026-09-14 17:48:00|disconnect|391             |ORPHANED_MATCH_EVENT   |
+--------+--------+-------------------+----------+----------------+-----------------------+



In [105]:
# checking if the player in the stat belongs to either
# of the team_a or team_b

In [107]:
#  Bring player team information into the player stats
df_stats_with_player_team = df_player_stats_valid.join(
    df_players_valid.select("player_id", "team_id").withColumnRenamed("team_id", "player_team_id"),
    on="player_id",
    how="inner"
)

In [108]:
# Join with match details to get team_a_id and team_b_id
df_stats_team_alignment = df_stats_with_player_team.join(
    df_matches_processed.select("match_id", "team_a_id", "team_b_id", "game"),
    on="match_id",
    how="inner"
)

In [109]:
# Add a column identifying whether the player played for Team A, Team B, or Neither (Mismatch/Anomalous)
df_stats_team_alignment_checked = df_stats_team_alignment.withColumn(
    "side",
    F.when(F.col("player_team_id") == F.col("team_a_id"), "TEAM_A")
     .when(F.col("player_team_id") == F.col("team_b_id"), "TEAM_B")
     .otherwise("MISMATCH_OR_UNKNOWN_TEAM")
)

In [110]:
# Check the distribution to see if any players recorded stats for teams they aren't on!
print("=== Player Side Distribution ===")
df_stats_team_alignment_checked.groupBy("side").count().show(truncate=False)

# Inspect any mismatched records if they exist
df_stats_team_alignment_checked.filter(
    F.col("side") == "MISMATCH_OR_UNKNOWN_TEAM"
).select("stat_id", "match_id", "player_id", "player_team_id", "team_a_id", "team_b_id").show(truncate=False)

=== Player Side Distribution ===
+------------------------+-----+
|side                    |count|
+------------------------+-----+
|TEAM_B                  |207  |
|MISMATCH_OR_UNKNOWN_TEAM|1270 |
|TEAM_A                  |198  |
+------------------------+-----+

+-------+--------+---------+--------------+---------+---------+
|stat_id|match_id|player_id|player_team_id|team_a_id|team_b_id|
+-------+--------+---------+--------------+---------+---------+
|ST00010|MT0054  |PL060    |TM01          |TM05     |TM07     |
|ST00053|MT0096  |PL034    |TM08          |TM06     |TM03     |
|ST00208|MT0077  |PL066    |TM07          |TM06     |TM08     |
|ST00684|MT0346  |PL063    |TM01          |TM04     |TM05     |
|ST01122|MT0206  |PL066    |TM07          |TM01     |TM04     |
|ST01205|MT0191  |PL050    |TM06          |TM03     |TM07     |
|ST01230|MT0029  |PL024    |TM08          |TM07     |TM01     |
|ST01382|MT0092  |PL024    |TM08          |TM03     |TM06     |
|ST01535|MT0099  |PL045    |TM0

In [111]:
# This is huge these are the stats that contains
# players whose actual team does not match 
# with the teams that are in that particular match
# so, these stats are not acceptable to include in our decision
# so, I am qurantining them

In [112]:
# 1. Isolate the mismatched records into a quarantine DataFrame
df_player_stats_side_quarantine = df_stats_team_alignment_checked.filter(
    F.col("side") == "MISMATCH_OR_UNKNOWN_TEAM"
).withColumn("stat_quarantine_reason", F.lit("PLAYER_TEAM_MISMATCH"))

# 2. Keep only the valid team-aligned records for your final analysis
df_player_stats_side_valid = df_stats_team_alignment_checked.filter(
    F.col("side").isin("TEAM_A", "TEAM_B")
)

# 3. Print the validation counts
print(f"Total evaluated player stats: {df_stats_team_alignment_checked.count()}")
print(f"Quarantined team-mismatch stats: {df_player_stats_side_quarantine.count()}")
print(f"Valid team-aligned stats: {df_player_stats_side_valid.count()}")

Total evaluated player stats: 1675
Quarantined team-mismatch stats: 1270
Valid team-aligned stats: 405


In [114]:
# checking the players having stats on cancelled matches
# Join valid player stats with match status information
df_stats_with_match_status = df_player_stats_valid.join(
    df_matches_processed.select("match_id", "match_status", "dq_status"),
    on="match_id",
    how="inner"
)

# Check the distribution of player stats by match status
print("=== Player Stats by Match Status ===")
df_stats_with_match_status.groupBy("match_status", "dq_status").count().orderBy(F.col("count").desc()).show(truncate=False)

=== Player Stats by Match Status ===
+------------+---------------------+-----+
|match_status|dq_status            |count|
+------------+---------------------+-----+
|completed   |VALID                |1056 |
|cancelled   |VALID                |341  |
|forfeit     |VALID                |294  |
|completed   |FLAGGED_TBD_TIMESTAMP|4    |
+------------+---------------------+-----+



In [115]:
# as cancelled matches stats are not required for analysis
# They might disturb the analysis
# I am quaranting them

In [116]:
# 1. Isolate player stats tied to cancelled matches into quarantine
df_player_stats_cancelled_quarantine = df_stats_with_match_status.filter(
    F.col("match_status") == "cancelled"
).withColumn("stat_quarantine_reason", F.lit("CANCELLED_MATCH_STATS"))

# 2. Keep only stats from completed or forfeit matches for your clean analytical base
df_player_stats_valid_final = df_stats_with_match_status.filter(
    F.col("match_status").isin("completed", "forfeit")
)

# Print the final validation counts
print(f"Total stats before match status filter: {df_stats_with_match_status.count()}")
print(f"Quarantined cancelled match stats: {df_player_stats_cancelled_quarantine.count()}")
print(f"Final valid stats base: {df_player_stats_valid_final.count()}")

Total stats before match status filter: 1695
Quarantined cancelled match stats: 341
Final valid stats base: 1354


In [128]:
# chcking wether each match_event is reffering to valid match and time period
# 1. Join df_match_events_processed with match start times
df_events_with_duration_check = df_match_events_processed.join(
    df_matches_processed.select("match_id", "parsed_match_time"),
    on="match_id",
    how="inner"
)

# 2. Safely convert duration_seconds to integer using try_cast and calculate match end time
df_event_duration_bounds = df_events_with_duration_check.withColumn(
    "clean_duration", F.expr("try_cast(duration_seconds AS INT)")
).withColumn(
    "match_end_time",
    F.when(
        F.col("clean_duration").isNotNull(),
        F.expr("from_unixtime(unix_timestamp(parsed_match_time) + clean_duration)")
    ).otherwise(None)
).withColumn(
    "parsed_match_end_time", F.col("match_end_time").cast("timestamp")
)

# 3. Check if event falls between match start and match end
df_event_window_check = df_event_duration_bounds.withColumn(
    "window_plausibility",
    F.when(
        F.col("parsed_match_end_time").isNull(), "DURATION_MISSING_OR_INVALID"
    ).when(
        (F.col("parsed_event_time") >= F.col("parsed_match_time")) & 
        (F.col("parsed_event_time") <= F.col("parsed_match_end_time")),
        "WITHIN_MATCH_WINDOW"
    ).otherwise("OUTSIDE_MATCH_WINDOW")
)

# 4. View distribution
print("=== Event Window Plausibility Distribution ===")
df_event_window_check.groupBy("window_plausibility").count().show(truncate=False)

=== Event Window Plausibility Distribution ===
+---------------------------+-----+
|window_plausibility        |count|
+---------------------------+-----+
|OUTSIDE_MATCH_WINDOW       |969  |
|DURATION_MISSING_OR_INVALID|30   |
+---------------------------+-----+



In [122]:
# here, again I found some data that is very doubtful
# becuase these events happened before the match time
# so, these records need further investigation
# so, I will quarantine these records

In [123]:
# 1. Isolate implausible pre-match events into quarantine
df_match_events_time_quarantine = df_event_time_check.filter(
    F.col("time_plausibility") == "IMPLAUSIBLE_PRE_MATCH_EVENT"
).withColumn("event_quarantine_reason", F.lit("IMPLAUSIBLE_PRE_MATCH_EVENT_TIME"))

# 2. Keep only plausibly timed events for downstream analysis
df_match_events_time_valid = df_event_time_check.filter(
    F.col("time_plausibility") == "PLAUSIBLE_TIMING"
)

# 3. Print validation counts
print(f"Total events checked: {df_event_time_check.count()}")
print(f"Quarantined implausible time events: {df_match_events_time_quarantine.count()}")
print(f"Valid timed events base: {df_match_events_time_valid.count()}")

Total events checked: 999
Quarantined implausible time events: 515
Valid timed events base: 484


In [129]:
# checking event timestamps and match timestamps
# 1. Join df_match_events_processed with match start times
df_events_with_duration_check = df_match_events_processed.join(
    df_matches_processed.select("match_id", "parsed_match_time"),
    on="match_id",
    how="inner"
)

# 2. Safely convert duration_seconds to integer using try_cast and calculate match end time
df_event_duration_bounds = df_events_with_duration_check.withColumn(
    "clean_duration", F.expr("try_cast(duration_seconds AS INT)")
).withColumn(
    "match_end_time",
    F.when(
        F.col("clean_duration").isNotNull(),
        F.expr("from_unixtime(unix_timestamp(parsed_match_time) + clean_duration)")
    ).otherwise(None)
).withColumn(
    "parsed_match_end_time", F.col("match_end_time").cast("timestamp")
)

# 3. Check if event falls between match start and match end
df_event_window_check = df_event_duration_bounds.withColumn(
    "window_plausibility",
    F.when(
        F.col("parsed_match_end_time").isNull(), "DURATION_MISSING_OR_INVALID"
    ).when(
        (F.col("parsed_event_time") >= F.col("parsed_match_time")) & 
        (F.col("parsed_event_time") <= F.col("parsed_match_end_time")),
        "WITHIN_MATCH_WINDOW"
    ).otherwise("OUTSIDE_MATCH_WINDOW")
)

# 4. View distribution
print("=== Event Window Plausibility Distribution ===")
df_event_window_check.groupBy("window_plausibility").count().show(truncate=False)

=== Event Window Plausibility Distribution ===
+---------------------------+-----+
|window_plausibility        |count|
+---------------------------+-----+
|OUTSIDE_MATCH_WINDOW       |969  |
|DURATION_MISSING_OR_INVALID|30   |
+---------------------------+-----+



In [130]:
# This again is a very importanat observation
# I could see the complete dataset match_events is heavily compromised
# This definitely needs an investigation
# so, quarantining the complete match_events dataset

In [131]:
# match score reconcillation
# 1. Classify the score-based outcome for each match
df_match_outcome_check = df_matches_processed.withColumn(
    "score_relationship",
    F.when(F.col("clean_team_a_score") > F.col("clean_team_b_score"), "TEAM_A_WIN")
     .when(F.col("clean_team_a_score") < F.col("clean_team_b_score"), "TEAM_B_WIN")
     .when(F.col("clean_team_a_score") == F.col("clean_team_b_score"), "DRAW")
     .otherwise("SCORE_MISSING_OR_INVALID")
)

# 2. Cross-reference score relationship with match_status
print("=== Match Status vs Score Relationship Distribution ===")
df_match_outcome_check.groupBy("match_status", "score_relationship").count().orderBy("match_status", "score_relationship").show(truncate=False)

# 3. Check for rule violations (e.g., completed matches with missing scores, or draws in forfeit/completed if draws aren't allowed)
df_score_anomalies = df_match_outcome_check.filter(
    (F.col("match_status") == "completed") & (F.col("score_relationship") == "SCORE_MISSING_OR_INVALID")
)
print(f"Completed matches with missing/invalid scores: {df_score_anomalies.count()}")

=== Match Status vs Score Relationship Distribution ===
+------------+------------------------+-----+
|match_status|score_relationship      |count|
+------------+------------------------+-----+
|cancelled   |DRAW                    |24   |
|cancelled   |SCORE_MISSING_OR_INVALID|5    |
|cancelled   |TEAM_A_WIN              |28   |
|cancelled   |TEAM_B_WIN              |24   |
|completed   |DRAW                    |52   |
|completed   |SCORE_MISSING_OR_INVALID|8    |
|completed   |TEAM_A_WIN              |93   |
|completed   |TEAM_B_WIN              |100  |
|forfeit     |DRAW                    |20   |
|forfeit     |SCORE_MISSING_OR_INVALID|4    |
|forfeit     |TEAM_A_WIN              |25   |
|forfeit     |TEAM_B_WIN              |17   |
+------------+------------------------+-----+

Completed matches with missing/invalid scores: 8


In [132]:
# 1. Isolate completed matches with missing or invalid scores into quarantine
df_matches_score_quarantine = df_match_outcome_check.filter(
    (F.col("match_status") == "completed") & (F.col("score_relationship") == "SCORE_MISSING_OR_INVALID")
).withColumn("match_quarantine_reason", F.lit("COMPLETED_MATCH_MISSING_SCORE"))

# 2. Keep valid completed matches with clean scores for analytics
df_matches_score_valid = df_match_outcome_check.filter(
    ~(
        (F.col("match_status") == "completed") & (F.col("score_relationship") == "SCORE_MISSING_OR_INVALID")
    )
)

# 3. Print validation counts
print(f"Total matches evaluated: {df_match_outcome_check.count()}")
print(f"Quarantined completed matches (missing score): {df_matches_score_quarantine.count()}")
print(f"Valid matches remaining: {df_matches_score_valid.count()}")

Total matches evaluated: 400
Quarantined completed matches (missing score): 8
Valid matches remaining: 392
